# Module 4: pandas I, working with tables

**Goal:** load a real dataset, look around, select and filter what you need, create new columns, and summarise by group.

**You will learn:** `read_csv` · `head/info/describe` · selecting columns and rows · filtering · sorting ·
new columns · `groupby` · missing values (first look)

**Maths in this module:** mean · median · variance and standard deviation · quantiles/percentiles · why the mean can mislead

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import numpy as np
import pandas as pd

students = pd.read_csv(DATA / "students.csv")

## 1. Looking around

pandas' main object is the **DataFrame**: a table where each column is a **Series** with its own type.
Whenever you load data, do these four things first.

In [ ]:
students.head()          # first 5 rows

In [ ]:
print(students.shape)            # (rows, columns)
print(students.columns.tolist())
students.info()                  # column types and non-missing counts

In [ ]:
students.describe()              # summary statistics of numeric columns

## 2. Selecting columns

`df["col"]` gives a Series; `df[["a", "b"]]` (double brackets) gives a smaller DataFrame.

In [ ]:
scores = students["exam_score"]
print(type(scores))
print(scores.mean(), scores.max())

students[["student_id", "exam_score"]].head(3)

## 3. Selecting rows: `loc` and `iloc`

* `iloc` selects by **position** (0, 1, 2, …)
* `loc` selects by **label** (index label / column name)

In [ ]:
print(students.iloc[0])                          # first row
print(students.iloc[0:3, 0:3])                   # first 3 rows, first 3 columns
print(students.loc[0:2, ["major", "exam_score"]])   # NOTE: loc includes the end label

## 4. Filtering with conditions

Exactly like NumPy masks: build a True/False Series, then use it to keep rows.
Combine conditions with `&` (and), `|` (or), `~` (not), **each wrapped in brackets**.

In [ ]:
high = students[students["exam_score"] >= 80]
print(len(high), "students scored 80+")

engineers_with_job = students[(students["major"] == "Engineering") & (students["part_time_job"] == "yes")]
print(len(engineers_with_job))

students[students["major"].isin(["Arts", "Business"])].head(3)

## 5. Sorting

In [ ]:
students.sort_values("exam_score", ascending=False).head(5)

In [ ]:
students.nlargest(3, "study_hours_week")[["student_id", "study_hours_week", "exam_score"]]

## 6. New columns

Assign to a new column name, using vectorised maths just like NumPy. Work on a `.copy()` when you
want to keep the original untouched.

In [ ]:
df = students.copy()

df["passed"] = df["exam_score"] >= 50
df["study_per_day"] = df["study_hours_week"] / 7
df["score_band"] = pd.cut(df["exam_score"], bins=[0, 50, 65, 80, 100],
                          labels=["fail", "ok", "good", "great"], include_lowest=True)

df[["exam_score", "passed", "study_per_day", "score_band"]].head()

## 7. Counting categories

In [ ]:
print(df["major"].value_counts())
print(df["part_time_job"].value_counts(normalize=True).round(2))    # proportions
print(df["score_band"].value_counts().sort_index())

## 8. `groupby`: split → apply → combine

"For each ___, compute ___" is a `groupby`. It is the most useful pandas idea for answering **comparison** questions.

In [ ]:
print(df.groupby("part_time_job")["exam_score"].mean().round(1))

In [ ]:
summary = df.groupby("major").agg(
    students=("student_id", "count"),
    mean_score=("exam_score", "mean"),
    median_score=("exam_score", "median"),
    mean_study=("study_hours_week", "mean"),
).round(1)
summary

## 🧮 9. Maths link: centre and spread

A single number can't describe a column. We usually give a **centre** and a **spread**.

| idea | formula / meaning | pandas |
|---|---|---|
| mean | $\bar{x} = \frac1n\sum x_i$ | `.mean()` |
| median | middle value when sorted | `.median()` |
| variance | $s^2 = \frac{1}{n-1}\sum (x_i-\bar{x})^2$ | `.var()` |
| standard deviation | $s = \sqrt{s^2}$, in the same units as the data | `.std()` |
| quantile / percentile | value below which a given share of data falls | `.quantile(0.9)` |

> Note: pandas' `.var()` and `.std()` divide by $n-1$ (the *sample* version). NumPy's divide by $n$ by default.
> With 200 rows the difference is tiny, but it's good to know it exists.

In [ ]:
s = df["exam_score"]
print("mean  ", s.mean().round(2))
print("median", s.median())
print("std   ", s.std().round(2))
print("middle 50% of scores are between", s.quantile(0.25), "and", s.quantile(0.75))
print("top 10% start at", s.quantile(0.9))

### The mean can mislead

Imagine ten salaries, then one billionaire walks in.

In [ ]:
salaries = pd.Series([30, 32, 35, 38, 40, 41, 43, 45, 48, 50])   # thousands
print("before: mean", salaries.mean(), " median", salaries.median())

with_outlier = pd.concat([salaries, pd.Series([5000])], ignore_index=True)
print("after:  mean", round(with_outlier.mean(), 1), " median", with_outlier.median())

The **mean** is dragged toward extreme values; the **median** barely moves. When data is skewed (incomes, house prices,
followers…), report the median, or both.

## 10. Missing values: a first look

Real data has gaps, shown as `NaN` ("not a number"). Our rides data has a few missing temperatures.

In [ ]:
rides = pd.read_csv(DATA / "daily_rides.csv")
print(rides["temp_c"].isna().sum(), "missing temperatures")
rides[rides["temp_c"].isna()]

In [ ]:
# Most pandas statistics skip NaN automatically:
print(rides["temp_c"].mean().round(2), "(NaN ignored)")
print(len(rides), "rows ->", rides["temp_c"].count(), "non-missing")

# Two simple (but not always right) options:
print(len(rides.dropna()), "rows after dropna()")
print(rides["temp_c"].fillna(rides["temp_c"].median()).isna().sum(), "missing after fillna(median)")

> Deciding *how* to handle missing data is a judgement call, and it's the first topic of Module 5.

## 11. Dates

The `date` column loaded as text. Convert it so pandas understands dates:

In [ ]:
rides["date"] = pd.to_datetime(rides["date"])
rides["month"] = rides["date"].dt.month
rides["weekday"] = rides["date"].dt.day_name()

rides.groupby("month")["rides"].mean().round(0)

In [ ]:
rides.groupby("is_weekend")["rides"].agg(["mean", "median", "count"]).round(1)

## 12. A first quick plot

Full visualisation comes in Module 6, but `.plot()` is a handy way to look at things right now.

In [ ]:
rides.groupby("month")["rides"].mean().plot(kind="bar", title="Average daily rides by month");

## Recap

* `head/info/describe` → `df["col"]`, `loc/iloc` → filters with `&`/`|` → `sort_values` → new columns → `groupby`
* Report centre **and** spread; the median resists outliers
* Converting types (`to_datetime`) unlocks new features (`.dt.month`)

**Next:** `exercises.ipynb`, then [`math-notes.md`](math-notes.md), then the **project checkpoint** in the module README.